# Final PCR Prediction (Classification)

In [1]:
import pandas as pd
import numpy as np

import joblib, csv

### Preprocessing

#### Before Preprocessing

In [2]:
dataset_x_all = pd.read_csv("../datasets/FinalTestDataset2025.csv")
dataset_x_all

,ID,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,TRG002219,47.0,1,1,0,0,3,2,1,1,...,0.498350,0.498350,3.144594,0.003447,8.257693e+06,150.048587,0.001753,0.037110,0.001369,0.001513
1,TRG002222,41.0,1,1,0,0,3,2,1,0,...,0.622381,0.622381,2.061654,0.006535,1.568442e+06,26.484938,0.009649,0.019352,0.000321,0.008285
2,TRG002223,53.0,0,0,0,1,2,1,1,1,...,0.412482,0.412482,3.440353,0.005391,2.656925e+06,174.606929,0.001594,0.075152,0.005255,0.001444
3,TRG002235,46.0,1,1,0,0,2,1,1,1,...,0.378333,0.378333,3.531715,0.007102,1.714787e+06,96.787378,0.002772,0.053377,0.002666,0.002406
4,TRG002240,39.0,0,0,1,0,2,2,1,1,...,0.524767,0.524767,2.186214,0.007896,5.104793e+05,12.789071,0.020072,0.023140,0.000463,0.017172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,TRG002953,62.3,0,0,1,0,2,1,1,0,...,0.550099,0.550097,2.632664,0.004373,1.583970e+06,91.084710,0.002972,0.042656,0.002168,0.002828
129,TRG002955,46.1,0,0,0,1,3,3,1,1,...,0.439568,0.439568,3.056046,0.001339,1.671271e+07,79.989003,0.003282,0.024716,0.000812,0.003078
130,TRG002956,44.3,0,0,0,1,3,2,1,0,...,0.450758,0.450758,2.820884,0.003774,4.493563e+06,144.506524,0.001912,0.053334,0.003453,0.001759
131,TRG002957,37.1,0,0,0,1,3,3,1,1,...,0.384860,0.384860,3.680654,0.003699,4.054944e+06,250.108074,0.001168,0.091763,0.008428,0.001107


#### Type-specific preprocessing
1. Drop ID
2. Change 'original_shape_VoxelVolume' to float
3. Get categorical/continuous column names
4. Replace 999 with NaN

In [3]:
ids = dataset_x_all["ID"]
dataset_x_all = dataset_x_all.drop("ID",axis=1)

dataset_x_all['original_shape_VoxelVolume'] = dataset_x_all['original_shape_VoxelVolume'].astype(np.float64)

categorical_names = dataset_x_all.select_dtypes(np.int64).columns
continuous_names = dataset_x_all.select_dtypes(np.float64).columns

dataset_x_all = dataset_x_all.replace([999], np.nan)

#### General Preprocessing
1. IQR Outlier Removal
2. Cap categorical data range (based on definitional range)
3. KNN Imputation
4. Z-scaling / Normalisation
5. Convert Categorical float columns back to int

In [4]:
Xs = dataset_x_all.copy()

lower_quartile = Xs[continuous_names].quantile(0.25)
upper_quartile = Xs[continuous_names].quantile(0.75)
iqr = upper_quartile - lower_quartile

lower_bound = lower_quartile - 1.5 * iqr
upper_bound = upper_quartile + 1.5 * iqr

Xs[continuous_names] = np.where(Xs[continuous_names] < lower_bound, lower_bound, Xs[continuous_names])
Xs[continuous_names] = np.where(Xs[continuous_names] > upper_bound, upper_bound, Xs[continuous_names])

In [5]:
categorical_ranges = {
    "pCR (outcome)": (0,1),
    "ER": (0, 1),
    "PgR": (0, 1),
    "HER2": (0, 1),
    "TrippleNegative": (0, 1),
    "ChemoGrade": (1, 3),
    "Proliferation": (1, 3),
    "HistologyType": (1, 2),
    "LNStatus": (0, 1),
    "TumourStage": (1, 4),
    "Gene": (0, 1)
}

In [6]:
data_train_cat = pd.concat([Xs[categorical_names]], axis=1)
for col in data_train_cat.columns:
    
    # Truncate data to be within range for categorical features
    data_train_cat[col] = np.where(data_train_cat[col] > categorical_ranges[col][1], categorical_ranges[col][1],  data_train_cat[col])
    data_train_cat[col] = np.where(data_train_cat[col] < categorical_ranges[col][0], categorical_ranges[col][0],  data_train_cat[col])

In [7]:
imputer_knn = joblib.load('../exports/pcr_imputer_knn.pkl')
z_scaler = joblib.load('../exports/pcr_z_scaler_knn.pkl')

Xs = pd.DataFrame(imputer_knn.transform(Xs), columns=Xs.columns, index=Xs.index)
Xs[continuous_names] = z_scaler.transform(Xs[continuous_names])
Xs

,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,TumourStage,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,-0.498161,1.0,1.0,0.0,0.0,3.0,2.0,1.0,1.0,2.0,...,0.672575,0.716197,0.582200,0.017874,0.234785,0.309580,-0.782707,-0.531875,-0.731600,-0.783273
1,-1.079838,1.0,1.0,0.0,0.0,3.0,2.0,1.0,0.0,2.0,...,1.491459,1.561724,-0.994303,1.328358,-0.644058,-0.756961,0.105371,-0.942832,-0.909987,0.241098
2,0.083516,0.0,0.0,0.0,1.0,2.0,1.0,1.0,1.0,2.0,...,0.105651,0.130828,1.012755,0.842922,-0.501052,0.521555,-0.800534,0.348486,-0.070205,-0.793813
3,-0.595107,1.0,1.0,0.0,0.0,2.0,1.0,1.0,1.0,3.0,...,-0.119810,-0.101968,1.145758,1.568689,-0.624831,-0.150145,-0.668088,-0.155443,-0.510835,-0.648251
4,-1.273730,0.0,0.0,1.0,0.0,2.0,2.0,1.0,1.0,2.0,...,0.846988,0.896285,-0.812973,1.905786,-0.783055,-0.875177,1.277726,-0.855181,-0.885904,1.585459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,0.985114,0.0,0.0,1.0,0.0,2.0,1.0,1.0,0.0,3.0,...,1.014236,1.068963,-0.163047,0.410887,-0.642018,-0.199368,-0.645590,-0.403547,-0.595612,-0.584466
129,-0.585412,0.0,0.0,0.0,1.0,3.0,3.0,1.0,1.0,4.0,...,0.284480,0.315475,0.453296,-0.877093,1.345618,-0.295140,-0.610688,-0.818701,-0.826407,-0.546559
130,-0.759915,0.0,0.0,0.0,1.0,3.0,2.0,1.0,0.0,2.0,...,0.358357,0.391756,0.110956,0.156474,-0.259751,0.261743,-0.764742,-0.156432,-0.376939,-0.746082
131,-1.457927,0.0,0.0,0.0,1.0,3.0,3.0,1.0,1.0,3.0,...,-0.076721,-0.057477,1.362577,0.124464,-0.317378,1.173244,-0.848443,0.732889,0.470020,-0.844800


In [8]:
# Convert categoricals back to int64 (no more NaN/float64) --------------------------------------------------
Xs[categorical_names] = Xs[categorical_names].astype(np.int64)
Xs

,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,TumourStage,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,-0.498161,1,1,0,0,3,2,1,1,2,...,0.672575,0.716197,0.582200,0.017874,0.234785,0.309580,-0.782707,-0.531875,-0.731600,-0.783273
1,-1.079838,1,1,0,0,3,2,1,0,2,...,1.491459,1.561724,-0.994303,1.328358,-0.644058,-0.756961,0.105371,-0.942832,-0.909987,0.241098
2,0.083516,0,0,0,1,2,1,1,1,2,...,0.105651,0.130828,1.012755,0.842922,-0.501052,0.521555,-0.800534,0.348486,-0.070205,-0.793813
3,-0.595107,1,1,0,0,2,1,1,1,3,...,-0.119810,-0.101968,1.145758,1.568689,-0.624831,-0.150145,-0.668088,-0.155443,-0.510835,-0.648251
4,-1.273730,0,0,1,0,2,2,1,1,2,...,0.846988,0.896285,-0.812973,1.905786,-0.783055,-0.875177,1.277726,-0.855181,-0.885904,1.585459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,0.985114,0,0,1,0,2,1,1,0,3,...,1.014236,1.068963,-0.163047,0.410887,-0.642018,-0.199368,-0.645590,-0.403547,-0.595612,-0.584466
129,-0.585412,0,0,0,1,3,3,1,1,4,...,0.284480,0.315475,0.453296,-0.877093,1.345618,-0.295140,-0.610688,-0.818701,-0.826407,-0.546559
130,-0.759915,0,0,0,1,3,2,1,0,2,...,0.358357,0.391756,0.110956,0.156474,-0.259751,0.261743,-0.764742,-0.156432,-0.376939,-0.746082
131,-1.457927,0,0,0,1,3,3,1,1,3,...,-0.076721,-0.057477,1.362577,0.124464,-0.317378,1.173244,-0.848443,0.732889,0.470020,-0.844800


Notes: HistologyType 1.5????????????????????????????????????????????????????????????????????????? LMAO

### Feature Selection
Currently selected method: Recursive Feature Elimination | 10 Features

In [9]:
feature_sel = joblib.load('../exports/selected_features.pkl')
feature_sel

['ER',
 'PgR',
 'TrippleNegative',
 'LNStatus',
 'Gene',
 'original_firstorder_InterquartileRange',
 'original_firstorder_MeanAbsoluteDeviation',
 'original_glrlm_LongRunLowGrayLevelEmphasis',
 'original_glszm_SmallAreaHighGrayLevelEmphasis',
 'original_glszm_SmallAreaLowGrayLevelEmphasis',
 'HER2']

In [10]:
Xs = Xs[feature_sel]
Xs

,ER,PgR,TrippleNegative,LNStatus,Gene,original_firstorder_InterquartileRange,original_firstorder_MeanAbsoluteDeviation,original_glrlm_LongRunLowGrayLevelEmphasis,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,HER2
0,1,1,0,1,1,-1.012475,-1.075555,1.804582,0.672575,0.716197,0
1,1,1,0,0,1,-0.924360,-1.052447,0.342037,1.491459,1.561724,0
2,0,0,1,1,1,-0.130771,-0.052262,0.068091,0.105651,0.130828,0
3,1,1,0,1,0,-0.089663,-0.074279,-0.534664,-0.119810,-0.101968,0
4,0,0,0,1,0,0.083155,0.122470,-0.596646,0.846988,0.896285,1
...,...,...,...,...,...,...,...,...,...,...,...
128,0,0,0,0,1,1.196384,1.551383,-0.730354,1.014236,1.068963,1
129,0,0,1,1,1,0.000158,0.098588,-0.690048,0.284480,0.315475,0
130,0,0,1,0,0,-1.376904,-1.405569,1.168971,0.358357,0.391756,0
131,0,0,1,1,1,-0.273968,-0.236742,-0.252826,-0.076721,-0.057477,0


### Prediction
1. Load pickled models
2. Predict outcomes using chosen model
3. Concat ID column with outcomes
4. Write to csv

In [11]:
models: dict
models = joblib.load('../exports/pcr_models.pkl') # Load all models

for key in models.keys():
    print(f"{key}: {models[key].predict(Xs)}")

# Then choose a model ig
# LogReg seems to have greaatest acc in testing
    # use it as output

m = models['Logistic Regression']
m_results = m.predict(Xs)

# print(m_results)
# print(ids)


# write to csv using original ids (data order remains the same as no train/test split for example)
with open("../datasets/FinalTestDataset2025Predictions.csv", "w", encoding="utf-8", newline='') as file:
    write = pd.concat([ids, pd.DataFrame(m_results, columns=['pCR (outcome)'])], ignore_index=False, sort=False, axis=1)
    write.to_csv(file, index=False)
    
write

Logistic Regression: [0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 1 1 1 1 0 0 0 1 0 1 0 0 1
 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 1 1 0 0 0 0 0 0 1 1 0 0 1 1 1 1 0 0 1
 0 0 1 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0 1 0 1 0 1 0
 1 1 1 1 0 1 1 0 0 1 0 1 0 0 0 0 0 1 0 0 0 1]
Naive Bayes (Gaussian): [0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0 1 0 0 0 1 0 1 0 0 1
 1 1 0 0 1 0 0 1 0 0 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1 0 1 1 0 0 0 1 1 1 0 0 1
 1 0 1 0 0 0 0 1 1 1 0 1 1 0 1 0 0 0 0 1 0 0 0 0 0 1 1 0 0 1 0 1 0 1 0 1 0
 1 1 1 1 0 1 1 0 0 1 0 1 0 0 0 0 0 1 1 0 0 1]
KNN (k=3): [0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 1
 0 1 0 1 1 0 1 0 0 1 0 0 0 0 0 1 1 0 0 1 0 0 0 0 1 0 1 1 1 0 0 1 0 0 0 1 1
 1 0 1 1 1 0 0 0 1 1 0 1 1 0 1 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 0 1
 1 1 1 1 0 0 1 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1]
SVM Hard (linear, C=100): [0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 1 1 0 0 0 0 1 0 1 0 0 1
 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0

,ID,pCR (outcome)
0,TRG002219,0
1,TRG002222,0
2,TRG002223,0
3,TRG002235,0
4,TRG002240,1
...,...,...
128,TRG002953,1
129,TRG002955,0
130,TRG002956,0
131,TRG002957,0
